# DESI-Ib TF Secondary Target Selection

## sga_semimajor_ends
### Method to create the secondary targeting file for the mid-points on the semi-major axis of large galaxies (from the SGA) in the BGS sample

##### Author: Kelly Douglass (University of Rochester)

See `/project/projectdirs/desi/target/secondary/README` for output data model

### Target classes
1. **Mid-points on the major axis**
2. Mid-points on the minor axis
3. Points along the major axis
4. Points off-axis

In [2]:
from astropy.table import Table, Column
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord

import numpy as np

## Parameters

In [3]:
OVERRIDE = True
REF_EPOCH = 2015.5

#output_directory = '/project/projectdirs/desi/target/secondary/indata/'
output_directory = ''

## Target catalogs

[Siena Galaxy Atlas](https://www.legacysurvey.org/sga/sga2020/)

In [17]:
# Target catalog file names

#input_directory = '/Users/kellydouglass/Documents/Research/data/SGA/'
input_directory = '/global/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/' #'/global/cfs/cdirs/cosmo/data/sga/2020/'

input_filename = input_directory + 'SGA2025-beta-parent-refcat-v1.6.kd.fits'

hdul = fits.open(input_filename)
hdul.info()
print(hdul[1].header)
large_galaxies = Table(hdul[1].data)
# tractor_fits = Table(hdul[6].data)
hdul.close()

Filename: /global/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/SGA2025-beta-parent-refcat-v1.6.kd.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1                1 BinTableHDU     27   470625R x 8C   [1D, 1D, 1K, 1E, 1J, 1E, 1E, 1E]   
  2                1 BinTableHDU     39   0R x 1C   [0A]   
  3                1 BinTableHDU     16   32768R x 1C   [4A]   
  4                1 BinTableHDU     21   32767R x 1C   [4A]   
  5                1 BinTableHDU     28   7R x 1C   [8A]   
  6                1 BinTableHDU     14   470625R x 1C   [12A]   
XTENSION= 'BINTABLE' / FITS Binary Table Extension                              BITPIX  =                    8 / 8-bits character format                        NAXIS   =                    2 / Tables are 2-D char. array                     NAXIS1  =                   44 / Bytes in row                                   NAXIS2  =               470625 / no comment                

In [18]:
large_galaxies[:5]

ra,dec,ref_id,mag,fitmode,pa,ba,diam
float64,float64,int64,float32,int32,float32,float32,float32
219.9258023922458,-32.53004546468686,4176048,17.78,0,70.35528,0.5832305,0.55550754
219.89199876009113,-32.34299358156585,4176016,17.71,0,150.42897,0.4607132,0.50789976
219.9409044773007,-32.331297660068095,242036,17.84,0,134.88263,0.22073081,0.68223655
220.20921498175156,-32.0109310161516,243654,18.04,0,91.79849,0.7038806,0.507167
220.3623232697446,-31.994910572523846,4257822,15.4,0,10.682078,0.6169077,1.0668384


In [19]:
# sga2025_north_filename = input_directory + 'SGA2025-ellipse-v1.6-dr11-north.kd.fits'

# hdul = fits.open(sga2025_north_filename)
# hdul.info()
# print(hdul[1].data[:5])
# hdul.close()

## Set first priority: points at $0.4R_{26}$ on the major axis

## Calculate (ra, dec) of major axis mid-points

We set two fiber target locations at either end of the major axis at a distance of $0.4R(26)$, where the value of 0.4 was determined based on the results of the measurements taken during SV.

In [20]:
center_ra = large_galaxies['ra']   # degrees
center_dec = large_galaxies['dec'] # degrees

centers = SkyCoord(center_ra*u.deg, center_dec*u.deg)

phi = large_galaxies['pa']*u.deg
r26 = 0.5*large_galaxies['diam']*u.arcmin

x = 0.4 # Determined during SV

# Maximum distance along the semi-major axis from the center coordinate for our endpoints
delta_a = x*r26

# Target positions
fiber1 = centers.directional_offset_by(phi, delta_a)
fiber2 = centers.directional_offset_by(phi + 180*u.deg, delta_a)

fiber_ra = np.concatenate((fiber1.ra, fiber2.ra))
fiber_dec = np.concatenate((fiber1.dec, fiber2.dec))

### Write target list to file

In [21]:
N_targets = len(fiber_ra)

In [22]:
disk_edges = Table([Column(fiber_ra, name='RA'), 
                    Column(fiber_dec, name='DEC'), 
                    Column(np.zeros(N_targets, dtype='>f4'), name='PMRA'), 
                    Column(np.zeros(N_targets, dtype='>f4'), name='PMDEC'), 
                    Column(REF_EPOCH*np.ones(N_targets, dtype='>f4'), name='REF_EPOCH'),
                    Column(OVERRIDE*np.ones(N_targets, dtype='bool'), name='OVERRIDE')])

In [23]:
disk_edges.write(output_directory + 'sga2025_semimajor_ends.fits', format='fits')

## Target statistics

In [25]:
print('The number of SGA-2025 galaxies is', len(large_galaxies))
print('The number of targets is', N_targets)

sky_chunk_boolean = np.logical_and.reduce([disk_edges['RA'] > 150, disk_edges['RA'] < 250, 
                                           disk_edges['DEC'] > 0, disk_edges['DEC'] < 50])

num_targets_in_sky_chunk = np.sum(sky_chunk_boolean)

sky_area = 100*50

print('The number of SGA galaxies in this portion of the sky is', 
      0.5*num_targets_in_sky_chunk)
print('The number of SGA galaxies per square degree is', 
      0.5*num_targets_in_sky_chunk/sky_area)
print('The number of fiber placements in this portion of the sky is', 
      num_targets_in_sky_chunk)
print('The number of fiber placements per square degree is', 
      num_targets_in_sky_chunk/sky_area)

The number of SGA-2025 galaxies is 470625
The number of targets is 941250
The number of SGA galaxies in this portion of the sky is 93338.0
The number of SGA galaxies per square degree is 18.6676
The number of fiber placements in this portion of the sky is 186676
The number of fiber placements per square degree is 37.3352
